# 05. Approved baseline-preparation v2

This notebook is the execution record for the approved baseline-preparation v2 contract. It uses the source-level planning boundary only: it binds the pinned target, validates the explicit operation authority, and inspects the anchored baseline-cache group. It does not materialize a cache unless a later, separately visible action is selected.


## Authority and boundary

The owner promoted this contract from draft to approved. Its scope is only `baseline_cache_preparation`, and its `baseline_cache_contract_id` anchors it to the original seed-42 baseline cache identity. The first expected outcome is therefore often **no selected action**: valid existing caches must be reused rather than regenerated.

This notebook neither evaluates candidates nor writes an evidence packet. A selected materialization action would be shown explicitly before it could be executed; no cell in this notebook calls the materializer.


In [ ]:
from pathlib import Path
import sys


def locate_project_root(start: Path) -> Path:
    """Find the source-controlled project without trusting notebook cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'experiments/schema/experiment-spec-v1.schema.json').is_file():
            return directory
        nested = directory / 'carrot-tuning-system'
        if (nested / 'experiments/schema/experiment-spec-v1.schema.json').is_file():
            return nested
    raise RuntimeError('Could not locate carrot-tuning-system project root.')


PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(REPOSITORY_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT / 'src'))

from evaluator.runner import LocalInnerLoopAdapter, LocalRunRequest

ADAPTER = LocalInnerLoopAdapter()
REQUEST = LocalRunRequest(
    spec_path=PROJECT_ROOT / 'experiments/approved/carrot_investment/'
        'seed_opportunity_baseline_preparation_v2.json',
    target_root=REPOSITORY_ROOT.parent / 'kaggriculture',
    # This is the run root; the planner adds its baseline-caches child.
    cache_root=REPOSITORY_ROOT / 'var/carrot-tuning-system',
    report_directory=REPOSITORY_ROOT / 'var/carrot-tuning-system/reports',
)


## 1. Gate and inspect the exact approved cache group

This call runs schema, target, lifecycle, operation, and local-control gates; then it reads and validates only the planned cache files. It returns a frozen selection. It does not invoke `materialize_selected_baseline`, so a successful result is still inspection evidence rather than a new simulation.


In [ ]:
BASELINE_PREPARATION_SELECTION = ADAPTER.plan_baseline_preparation(REQUEST)
if BASELINE_PREPARATION_SELECTION.selected_action is not None:
    raise RuntimeError(
        'A materialization action was selected. Stop here for a visible, separate execution review.'
    )


In [ ]:
def render_baseline_preparation_result(selection) -> None:
    """Presentation only: render already-established cache-plan facts."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Approved baseline-preparation result

| Fact | Result |
|:---|:---|
| Contract | `{selection.specification['experiment_id']}` |
| Lifecycle | `{selection.lifecycle.declared_status}` |
| Operation | **{selection.operation.is_authorized}** |
| Reusable caches | `{len(selection.cache_preparation.reusable_actions)}` |
| Selected materialization action | **None** |
| Simulator invoked | **No** |

> Both anchored caches are reusable. This establishes cache-provenance continuity only; it is not candidate evidence or a policy decision.
"""
    ))


render_baseline_preparation_result(BASELINE_PREPARATION_SELECTION)


## Closing record

The approved v2 contract found the original seed-42 cache group reusable, so no materialization action existed to execute. No simulator run, cache write, candidate evaluation, packet write, or policy-source change occurred. The next decision is a distinct experiment envelope, not repeated baseline preparation.
